In [1]:
import os
os.environ["TORCHINDUCTOR_CACHE_DIR"] = "C:/temp/torch_cache"
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cpu


In [2]:
data_dir = r"C:\Users\PINTU SHAW\Downloads\archive (3)\chest_xray"

train_dir = os.path.join(data_dir, "train")
val_dir   = os.path.join(data_dir, "val")
test_dir  = os.path.join(data_dir, "test")

In [8]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])
train_data = datasets.ImageFolder(train_dir, transform=transform)
val_data   = datasets.ImageFolder(val_dir, transform=transform)
test_data  = datasets.ImageFolder(test_dir, transform=transform)

trainloader = DataLoader(train_data, batch_size=32, shuffle=True)
valloader   = DataLoader(val_data, batch_size=32)
testloader  = DataLoader(test_data, batch_size=32)

class_names = train_data.classes
num_classes = len(class_names)

print("Classes:", class_names)

Classes: ['NORMAL', 'PNEUMONIA']


In [5]:
import os
os.environ['TORCH_HOME'] = 'C:/torch_cache'
import torchvision.models as models
resnet_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

print("Original final layer:")
print(resnet_model.fc)

# Freeze feature layers
print("\nFreezing feature layers")
for param in resnet_model.parameters():
    param.requires_grad = False

print("Feature extraction layers frozen")

# Modify final layer
num_features = resnet_model.fc.in_features
resnet_model.fc = nn.Linear(num_features, 2)

print("\nNew final layer:")
print(resnet_model.fc)

resnet_model = resnet_model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer_resnet = optim.Adam(resnet_model.fc.parameters(), lr=0.001)

print("\nOnly classifier parameters will update")

Original final layer:
Linear(in_features=2048, out_features=1000, bias=True)

Freezing feature layers
Feature extraction layers frozen

New final layer:
Linear(in_features=2048, out_features=2, bias=True)

Only classifier parameters will update


In [9]:
def train_model(model, optimizer, name):
    epochs =2

    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        print(f"\n{name} - Epoch: {epoch+1}")

        model.train()

        for i, (images, labels) in enumerate(trainloader):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            if i % 100 == 0:
                print(f"Batch: {i}, Loss: {loss.item()}")

        epoch_loss = running_loss / len(trainloader)
        accuracy = 100 * correct / total

        print("Epoch loss:", epoch_loss)
        print("Training accuracy:", accuracy)

    print(f"\n{name} Training Finished")

In [10]:
train_model(resnet_model, optimizer_resnet, "ResNet50")


ResNet50 - Epoch: 1
Batch: 0, Loss: 0.6989907026290894
Batch: 100, Loss: 0.17043636739253998
Epoch loss: 0.28420995326678444
Training accuracy: 89.3213190184049

ResNet50 - Epoch: 2
Batch: 0, Loss: 0.2081492394208908
Batch: 100, Loss: 0.1808970868587494
Epoch loss: 0.16735729591155346
Training accuracy: 94.07592024539878

ResNet50 Training Finished


In [26]:
resnet_model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in valloader:
        images, labels = images.to(device), labels.to(device)
        outputs = resnet_model(images)

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Validation Accuracy:", 100 * correct / total)

Validation Accuracy: 74.46808510638297


In [17]:
torch.save(resnet_model.state_dict(), 
           r"C:\HospitalProject\models" + os.sep + "chest_xray_model.pth")

In [35]:
import torch
import torch.nn as nn
import torchvision.models as models
from PIL import Image
from torchvision import transforms

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 🔹 Load model (same as training)
model  = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

# 🔹 Load weights
model.load_state_dict(torch.load(r"C:\HospitalProject\models" + os.sep + "chest_xray_model.pth", map_location=device))
model.eval()

# 🔹 Class names
class_names = ["NORMAL", "PNEUMONIA"]

# ✅ IMPORTANT: Add normalization
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],   # ImageNet mean
        std=[0.229, 0.224, 0.225]     # ImageNet std
    )
])

# 🔹 Prediction function
def predict_image(img_path):
    image = Image.open(img_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]

# 🔹 Test
img_path = r"C:\Users\PINTU SHAW\Downloads\archive (3)\chest_xray\test\NORMAL\IM-0093-0001.jpeg"
print("Prediction:", predict_image(img_path))

Prediction: PNEUMONIA
